# 🛒 SupplyMe — notebook Google Colab

*Predict. Anticipate. Convert.*

Ce notebook exécute tout le projet SupplyMe sur les données **Instacart** :
1. connexion à Google Drive ;
2. copie du projet et décompression des données ;
3. installation des dépendances ;
4. pipeline complet : données → base SQL → prédictions → marketing → rapport ;
5. affichage des résultats (métriques, graphiques, requêtes SQL, panier client) ;
6. *(optionnel)* lancement de l'interface Streamlit.

**Avant de commencer**, ton Google Drive doit contenir :
```
Mon Drive/
└── SupplyMe_MVP/
    ├── SupplyMe.zip            ← le code du projet
    ├── SupplyMe_Colab.ipynb    ← ce notebook
    └── instacart/              ← les 6 fichiers Instacart (.csv ou .csv.zip)
```
Les résultats seront enregistrés dans `Mon Drive/SupplyMe_MVP/resultats/`.

➡️ Exécute les cellules **dans l'ordre**, avec ▶ ou `Maj + Entrée`.

## 1. Connexion à Google Drive
Une fenêtre s'ouvre : choisis ton compte Google puis clique sur **Autoriser**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Copie du projet et préparation des données
On travaille sur une copie locale (`/content/SupplyMe`) : c'est beaucoup plus rapide que de lire les gros fichiers directement sur le Drive.

In [ ]:
import os, glob, zipfile, shutil

DRIVE_DIR = '/content/drive/MyDrive/SupplyMe_MVP'    # ← modifie si ton dossier est ailleurs
LOCAL_PROJECT = '/content/SupplyMe'

assert os.path.exists(f'{DRIVE_DIR}/SupplyMe.zip'), f"SupplyMe.zip introuvable dans {DRIVE_DIR}"
if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)
with zipfile.ZipFile(f'{DRIVE_DIR}/SupplyMe.zip') as f:     # le zip contient le dossier SupplyMe/
    f.extractall('/content')

raw = f'{LOCAL_PROJECT}/data/raw'
for src in glob.glob(f'{DRIVE_DIR}/instacart/*'):           # copie + décompression des données
    if src.endswith('.zip'):
        with zipfile.ZipFile(src) as f:
            for name in f.namelist():
                if name.endswith('.csv') and not name.startswith('__MACOSX'):
                    with f.open(name) as s, open(f'{raw}/{os.path.basename(name)}', 'wb') as d:
                        shutil.copyfileobj(s, d)
    elif src.endswith('.csv'):
        shutil.copy(src, raw)

required = ['orders.csv', 'order_products__prior.csv', 'order_products__train.csv',
            'products.csv', 'aisles.csv', 'departments.csv']
for f in required:
    print(('✅' if os.path.exists(f'{raw}/{f}') else '❌'), f)
assert all(os.path.exists(f'{raw}/{f}') for f in required), "Il manque des fichiers Instacart dans SupplyMe_MVP/instacart."
os.chdir(LOCAL_PROJECT)
print('\nDossier de travail :', os.getcwd())

## 3. Installation des dépendances (≈ 1 min)

In [ ]:
!pip install -q -r requirements.txt

## 4. Pipeline complet (≈ 2 min)
`--n-customers` fixe la taille de l'échantillon de clients Instacart (3 000 par défaut).

In [ ]:
N_CUSTOMERS = 3000
!python main.py run --source instacart --n-customers {N_CUSTOMERS}

## 5. Tests unitaires
Vérifient l'absence de fuite temporelle et la logique des trois décisions marketing.

In [ ]:
!python -m pytest -q

## 6. Résultats
### 6.1 Qualité des données et performances des modèles

In [ ]:
import json, sys
import pandas as pd
from IPython.display import Image, Markdown, display
sys.path.insert(0, LOCAL_PROJECT)
from database.db import query, get_metadata

meta = get_metadata()
display(pd.Series({k.removeprefix('data_'): v for k, v in meta.items() if k.startswith('data_')}, name='valeur').to_frame())

m = json.load(open('prediction/artifacts/metrics.json'))
reg = pd.DataFrame({k: m[k] for k in ['baseline_mean', 'baseline_median', 'random_forest']}).T
display(reg[['MAE', 'RMSE', 'MedAE', 'within_3_days', 'within_7_days']].round(3))
print(f"Gain de MAE vs baseline médiane : {m['random_forest']['MAE_gain_vs_median']:.1%}")
print(f"Classifieur achat sous 7 jours : AUC = {m['rf_classifier_7d']['AUC']:.3f}")

### 6.2 Graphiques (réutilisables dans le rapport)

In [ ]:
for f in sorted(glob.glob('reports/figures/*.png')):
    print(os.path.basename(f))
    display(Image(f, width=700))

### 6.3 Rentabilité des stratégies marketing

In [ ]:
display(pd.read_csv('reports/benchmark_strategies.csv'))
display(query('''SELECT segment, decision, COUNT(*) AS n, ROUND(AVG(discount_rate), 3) AS reduction_moy,
                        ROUND(SUM(expected_profit), 2) AS profit
                 FROM campaigns GROUP BY segment, decision'''))

### 6.4 Exemples de requêtes SQL (habitudes d'achat)

In [ ]:
query('''
SELECT h.customer_id, p.product_name, p.category, h.n_purchases, h.mean_interval, h.last_purchase
FROM v_customer_product_habits h
JOIN products p ON p.product_id = h.product_id
WHERE h.n_purchases >= 5
ORDER BY h.n_purchases DESC
LIMIT 15
''')

### 6.5 Parcours d'un client : prédiction → décision → panier → message

In [ ]:
from marketing.basket import build_baskets, compose_message

summary = pd.read_csv('reports/baskets_summary.csv')
CLIENT = int(summary.sort_values('n_products', ascending=False)['customer_id'].iloc[0])  # ← ou un autre id
print('Client :', CLIENT)
display(query('''SELECT pr.product_name, p.last_purchase_date, p.predicted_date,
                        ROUND(p.predicted_interval, 1) AS dans_j, ROUND(p.purchase_proba, 3) AS p_achat_7j,
                        c.segment, c.decision, c.discount_rate, ROUND(c.expected_profit, 3) AS profit, c.campaign_date
                 FROM campaigns c
                 JOIN predictions p ON p.customer_id = c.customer_id AND p.product_id = c.product_id
                 JOIN products pr ON pr.product_id = c.product_id
                 WHERE c.customer_id = ?''', (CLIENT,)))
print(compose_message(build_baskets(CLIENT)))

## 7. Sauvegarde des résultats sur ton Drive
Copie la base, le rapport, les figures et les modèles dans `Mon Drive/SupplyMe_MVP/resultats/`.

In [ ]:
OUT = f'{DRIVE_DIR}/resultats'
os.makedirs(OUT, exist_ok=True)
for item in ['supplyme.db', 'reports', 'prediction/artifacts']:
    src, dst = f'{LOCAL_PROJECT}/{item}', f'{OUT}/{os.path.basename(item)}'
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    print('✅ sauvegardé :', dst)

## 8. *(Optionnel)* Interface Streamlit dans Colab

Colab ne peut pas afficher Streamlit directement : on crée un lien public temporaire avec **cloudflared**.
1. Exécute la cellule.
2. Attends la ligne `https://xxxx.trycloudflare.com` (≈ 20 s) et clique dessus.
3. Pour arrêter : bouton ⏹ de la cellule.

⚠️ Le lien est public tant que la cellule tourne : ne le partage qu'à ton groupe.

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup streamlit run app/streamlit_app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
!sleep 5 && ./cloudflared tunnel --url http://localhost:8501 2>&1 | grep --line-buffered -o 'https://[a-z0-9-]*\.trycloudflare\.com'